# PubMed

In [1]:
 # This code works in Python 3.10.6
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import networkx as nx
from torch_geometric.datasets.dblp import DBLP
import random
import torch
from torch import optim
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, Linear, SAGEConv
import csv
import random

import warnings
warnings.filterwarnings('ignore')

Load Dataset

In [1]:
#Download the PubMed dataset from https://github.com/yangji9181/HNE/tree/master/Data and reference the files node.dat,label.dat,label.dat.test in order to execute the code below

In [2]:
df_nodes = pd.read_table(('PubMed/node.dat'),names=['node_id', 'node_name', 'node_type', 'node_attributes'],quoting=csv.QUOTE_NONE)
df_nodes

,node_id,node_name,node_type,node_attributes
0,0,ITCC,2,"0.02508,-0.036716,0.091492,-0.074665,0.022286,..."
1,1,gamma-Butyrolactone,2,"-0.072251,-0.1242,0.027377,-0.049692,0.022852,..."
2,2,eIF4E,0,"-0.85981,-1.632455,-0.238721,-0.527147,1.09448..."
3,3,abietane,2,"-0.100137,-0.295236,0.204746,-0.043928,0.09525..."
4,4,DOGMATISM,2,"0.005788,-0.080936,-0.038131,0.039531,-0.00082..."
...,...,...,...,...
63104,63104,archival_tumor,1,"0.066291,0.020142,0.146529,-0.058354,0.11038,0..."
63105,63105,diflucortolone,2,"0.027386,-0.231205,0.110102,0.058394,-0.052956..."
63106,63106,A2BP1,0,"0.038467,-0.022038,0.046468,-0.087253,0.157027..."
63107,63107,Digital_Health_Literacy,1,"-0.01894,-0.061136,0.097672,-0.102349,0.078652..."


In [3]:
df_labels_train = pd.read_table(('PubMed/label.dat'),names=['node_id', 'node_name', 'node_type', 'node_label'])
df_labels_train['split'] = 'Train'

In [4]:
df_labels_test = pd.read_table(('PubMed/label.dat.test'),names=['node_id', 'node_name', 'node_type', 'node_label'])

In [5]:
df_labels_val = df_labels_test[:43]
df_labels_val['split'] = 'Val'

In [6]:
df_labels_test.drop(index=df_labels_test.index[:43], axis=0, inplace=True)
df_labels_test['split'] = 'Test'

In [7]:
df_labels = pd.concat([df_labels_train, df_labels_val, df_labels_test],ignore_index=True)

TYPE	MEANING
0		GENE
1		DISEASE
2		CHEMICAL
3		SPECIES

Data Preparation

In [8]:
df_disease = df_nodes[df_nodes['node_type'] == 1]
df_disease

,node_id,node_name,node_type,node_attributes
9,9,RVF,1,"-1.072947,1.158542,0.498745,-0.085264,0.474166..."
10,10,diarrheal_enterotoxin,1,"-0.016865,0.004677,0.066797,-0.062516,-0.0147,..."
13,13,VUSE,1,"-0.075162,0.044893,0.004554,-0.070527,-0.00537..."
14,14,cardiac_auscultation,1,"0.07281,-0.075333,-0.027466,0.005992,-0.065542..."
15,15,intraoral_squamous_cell_carcinoma,1,"0.043272,-0.057455,0.068631,0.059395,-0.065671..."
...,...,...,...,...
63095,63095,Schistosoma_mansoni_infections,1,"-0.013617,-0.046871,0.036176,-0.063161,-0.0984..."
63096,63096,permeability_transition_pore,1,"0.027475,0.017549,0.04185,0.013369,-0.092689,-..."
63098,63098,nutrient-deficient,1,"0.012721,-0.355395,-0.074993,-0.000885,-0.0688..."
63104,63104,archival_tumor,1,"0.066291,0.020142,0.146529,-0.058354,0.11038,0..."


In [9]:
df_disease=df_disease.merge(df_labels, how='left', on='node_id')[['node_id','node_attributes','node_label','split']]
df_disease

,node_id,node_attributes,node_label,split
0,9,"-1.072947,1.158542,0.498745,-0.085264,0.474166...",NaN,NaN
1,10,"-0.016865,0.004677,0.066797,-0.062516,-0.0147,...",NaN,NaN
2,13,"-0.075162,0.044893,0.004554,-0.070527,-0.00537...",NaN,NaN
3,14,"0.07281,-0.075333,-0.027466,0.005992,-0.065542...",NaN,NaN
4,15,"0.043272,-0.057455,0.068631,0.059395,-0.065671...",NaN,NaN
...,...,...,...,...
20158,63095,"-0.013617,-0.046871,0.036176,-0.063161,-0.0984...",NaN,NaN
20159,63096,"0.027475,0.017549,0.04185,0.013369,-0.092689,-...",NaN,NaN
20160,63098,"0.012721,-0.355395,-0.074993,-0.000885,-0.0688...",NaN,NaN
20161,63104,"0.066291,0.020142,0.146529,-0.058354,0.11038,0...",NaN,NaN


In [10]:
df_disease.fillna(-1,inplace=True)

In [11]:
df_disease['node_label'] = df_disease.node_label.astype(int)

In [12]:
df_disease

,node_id,node_attributes,node_label,split
0,9,"-1.072947,1.158542,0.498745,-0.085264,0.474166...",-1,-1
1,10,"-0.016865,0.004677,0.066797,-0.062516,-0.0147,...",-1,-1
2,13,"-0.075162,0.044893,0.004554,-0.070527,-0.00537...",-1,-1
3,14,"0.07281,-0.075333,-0.027466,0.005992,-0.065542...",-1,-1
4,15,"0.043272,-0.057455,0.068631,0.059395,-0.065671...",-1,-1
...,...,...,...,...
20158,63095,"-0.013617,-0.046871,0.036176,-0.063161,-0.0984...",-1,-1
20159,63096,"0.027475,0.017549,0.04185,0.013369,-0.092689,-...",-1,-1
20160,63098,"0.012721,-0.355395,-0.074993,-0.000885,-0.0688...",-1,-1
20161,63104,"0.066291,0.020142,0.146529,-0.058354,0.11038,0...",-1,-1


In [13]:
df_gene = df_nodes[df_nodes['node_type'] == 0]
df_gene

,node_id,node_name,node_type,node_attributes
2,2,eIF4E,0,"-0.85981,-1.632455,-0.238721,-0.527147,1.09448..."
5,5,UBLCP1,0,"-0.116854,-0.120575,0.07072,-0.166677,0.106602..."
6,6,nodZ,0,"-0.042955,-0.014979,0.023987,0.025399,0.013668..."
11,11,grp170,0,"-0.106744,-0.079176,-0.107495,0.049958,-0.0039..."
12,12,B6_and_D2,0,"-0.097067,-0.050246,0.000982,0.024265,0.025848..."
...,...,...,...,...
63090,63090,hCGalpha,0,"-0.189161,-0.110873,0.018683,0.030247,0.08659,..."
63093,63093,HSD2,0,"-0.03078,-0.16347,0.081172,0.023949,0.05953,-0..."
63099,63099,pgi1,0,"-0.021245,-0.040226,0.059328,-0.154712,-0.0274..."
63103,63103,cytotoxic_T-lymphocyte_antigen-4,0,"-0.027838,-0.221384,0.031655,0.041904,0.094765..."


In [14]:
df_chemical = df_nodes[df_nodes['node_type'] == 2]
df_chemical

,node_id,node_name,node_type,node_attributes
0,0,ITCC,2,"0.02508,-0.036716,0.091492,-0.074665,0.022286,..."
1,1,gamma-Butyrolactone,2,"-0.072251,-0.1242,0.027377,-0.049692,0.022852,..."
3,3,abietane,2,"-0.100137,-0.295236,0.204746,-0.043928,0.09525..."
4,4,DOGMATISM,2,"0.005788,-0.080936,-0.038131,0.039531,-0.00082..."
7,7,TZDs,2,"-0.383792,-0.101718,-0.271882,-0.411151,0.1531..."
...,...,...,...,...
63100,63100,Dibutyryl_cyclic_AMP,2,"-0.01441,-0.2465,-0.061486,-0.11033,-0.050258,..."
63101,63101,Ag3Sn,2,"0.062272,0.027182,-0.130906,-0.13105,-0.049692..."
63102,63102,PAL-LPC,2,"-0.017939,0.004023,0.050958,-0.115518,-0.12366..."
63105,63105,diflucortolone,2,"0.027386,-0.231205,0.110102,0.058394,-0.052956..."


In [15]:
df_species = df_nodes[df_nodes['node_type'] == 3]
df_species

,node_id,node_name,node_type,node_attributes
16,16,painted_turtles,3,"-0.095541,-0.209068,-0.167771,0.081688,0.23290..."
30,30,Rickettsia_rickettsii,3,"0.030015,-0.184471,0.088601,-0.097601,-0.07233..."
52,52,Ictalurus_punctatus,3,"-0.093925,-0.211232,0.173214,0.096446,-0.06752..."
128,128,N._GONORRHOEAE,3,"-0.014923,-0.060145,0.014826,-0.00509,0.009751..."
140,140,giraffes,3,"-0.174262,0.005391,0.191771,0.075813,0.00924,-..."
...,...,...,...,...
62965,62965,Methanospirillum_hungatei,3,"0.019382,-0.09156,0.09262,-0.020885,-0.007555,..."
62977,62977,H._pylori,3,"2.070796,-0.284317,2.720454,-0.739623,-1.43600..."
62998,62998,F_nucleatum,3,"-0.026027,-0.116689,0.046049,-0.065215,-0.0877..."
63031,63031,H._pylori_strain,3,"-0.024642,-0.024474,0.113648,-0.044572,-0.1329..."


In [16]:
def convert_string_to_float(df):
    return df['node_attributes'].apply(lambda x: np.fromstring(x, dtype=float, sep=',' ))

In [17]:
def convert_to_tensor(df):
    return torch.tensor(df).to(dtype=torch.float32)

In [18]:
disease= convert_string_to_float(df_disease)
x_disease = convert_to_tensor(disease)

In [19]:
y_disease = torch.tensor(np.array(df_disease['node_label']), dtype=torch.long)

In [20]:
gene = convert_string_to_float(df_gene)
gene = gene.reset_index(drop=True)
x_gene = convert_to_tensor(gene)

In [21]:
chemical = convert_string_to_float(df_chemical)
chemical = chemical.reset_index(drop=True)
x_chemical = convert_to_tensor(chemical)

In [22]:
species = convert_string_to_float(df_species)
species = species.reset_index(drop=True)
x_species = convert_to_tensor(species)

Create Hetero Data

In [23]:
data = HeteroData({'disease':{'x': x_disease, 'y':y_disease},'gene':{'x': x_gene},
                          'chemical':{'x': x_chemical},'species':{'x': x_species}})

In [24]:
df_edges = pd.read_table(('PubMed/link.dat'),names=['source', 'target', 'link_type', 'link_weight'])
df_edges

,source,target,link_type,link_weight
0,47789,32267,8,225
1,14228,31867,3,2
2,35405,31559,5,2
3,31559,35405,5,2
4,885,32267,8,474
...,...,...,...,...
236453,4079,62356,2,1
236454,30859,57440,2,1
236455,39493,62538,2,1
236456,39493,32267,2,1


In [25]:
#Get lists of edges
batchsize = 500
gene_to_gene = []
gene_to_disease = []
disease_to_disease = []
chemical_to_gene = []
chemical_to_disease = []
chemical_to_chemical = []
chemical_to_species = []
species_to_gene = []
species_to_disease = []
species_to_species = []
remaining_edges = []


for i in range(0, len(df_edges), batchsize):
    batch = df_edges[i:i+batchsize]

    if (batch.loc[i, "source"] in list(df_gene['node_id'])) and (batch.loc[i, "target"] in list(df_gene['node_id'])):
            gene_to_gene.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_gene['node_id'])) and (batch.loc[i, "target"] in list(df_disease['node_id'])):
            gene_to_disease.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_disease['node_id'])) and (batch.loc[i, "target"] in list(df_disease['node_id'])):
            disease_to_disease.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_chemical['node_id'])) and (batch.loc[i, "target"] in list(df_gene['node_id'])):
            chemical_to_gene.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_chemical['node_id'])) and (batch.loc[i, "target"] in list(df_disease['node_id'])):
            chemical_to_disease.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_chemical['node_id'])) and (batch.loc[i, "target"] in list(df_chemical['node_id'])):
            chemical_to_chemical.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_chemical['node_id'])) and (batch.loc[i, "target"] in list(df_species['node_id'])):
            chemical_to_species.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_species['node_id'])) and (batch.loc[i, "target"] in list(df_gene['node_id'])):
            species_to_gene.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_species['node_id'])) and (batch.loc[i, "target"] in list(df_disease['node_id'])):
            species_to_disease.append((batch.loc[i, "source"],batch.loc[i, "target"]))
            
    elif (batch.loc[i, "source"] in list(df_species['node_id'])) and (batch.loc[i, "target"] in list(df_species['node_id'])):
            species_to_species.append((batch.loc[i, "source"],batch.loc[i, "target"]))
    else:
        remaining_edges.append((batch.loc[i, "source"],batch.loc[i, "target"]))

In [26]:
remaining_edges

[]

In [27]:
def preprocess_edges(edgelist,node_list):
    res = [[node_list[i] for i, j in edgelist],[node_list[j] for i, j in edgelist]] 
    node_from = torch.tensor(res[0])
    node_to = torch.tensor(res[1])
    edges = torch.concat((node_from,node_to)).reshape(-1,len(node_from))
    return edges

In [28]:
def remap_indices(node_list):
    val_list = [*range(0, len(node_list), 1)]
    return dict(zip(node_list,val_list))

In [29]:
 #Re-map indices to correct range
gene_nodes_mapping = remap_indices(list(df_gene["node_id"]))
disease_nodes_mapping = remap_indices(list(df_disease["node_id"]))
chemical_nodes_mapping = remap_indices(list(df_chemical["node_id"]))
species_nodes_mapping = remap_indices(list(df_species["node_id"]))

In [30]:
node_list = {}
for d in [gene_nodes_mapping, disease_nodes_mapping, chemical_nodes_mapping,species_nodes_mapping]:
    node_list.update(d)

In [31]:
#Prepare edge tensor for hetero data
if gene_to_gene:
    edge_index_gene_gene = preprocess_edges(gene_to_gene,node_list)
    data['gene','to','gene'].edge_index = edge_index_gene_gene
if gene_to_disease:
    edge_index_gene_disease = preprocess_edges(gene_to_disease,node_list)
    data['gene','to','disease'].edge_index = edge_index_gene_disease
if disease_to_disease:
    edge_index_disease_disease = preprocess_edges(disease_to_disease,node_list)
    data['disease','to','disease'].edge_index = edge_index_disease_disease
if chemical_to_gene:
    edge_index_chemical_gene = preprocess_edges(chemical_to_gene,node_list)
    data['chemical','to','gene'].edge_index = edge_index_chemical_gene
if chemical_to_disease:
    edge_index_chemical_disease = preprocess_edges(chemical_to_disease,node_list)
    data['chemical','to','disease'].edge_index = edge_index_chemical_disease
if chemical_to_chemical:
    edge_index_chemical_chemical = preprocess_edges(chemical_to_chemical,node_list)
    data['chemical','to','chemical'].edge_index = edge_index_chemical_chemical
if chemical_to_species:
    edge_index_chemical_species = preprocess_edges(chemical_to_species,node_list)
    data['chemical','to','species'].edge_index = edge_index_chemical_species
if species_to_gene:
    edge_index_species_gene = preprocess_edges(species_to_gene,node_list)
    data['species','to','gene'].edge_index = edge_index_species_gene
if species_to_disease:
    edge_index_species_disease = preprocess_edges(species_to_disease,node_list)
    data['species','to','disease'].edge_index = edge_index_species_disease
if species_to_species:
    edge_index_species_species = preprocess_edges(species_to_species,node_list)
    data['species','to','species'].edge_index = edge_index_species_species

In [32]:
train_ids= df_disease.index[df_disease['split'] == 'Train'].tolist()
val_ids= df_disease.index[df_disease['split'] == 'Val'].tolist()
test_ids= df_disease.index[df_disease['split'] == 'Test'].tolist()

In [33]:
train_mask = torch.zeros(df_disease.shape[0], dtype=torch.bool)
val_mask = torch.zeros(df_disease.shape[0], dtype=torch.bool)
test_mask = torch.zeros(df_disease.shape[0], dtype=torch.bool)

for idx in train_ids:
    train_mask[idx] = True
    
for idx in val_ids:
    val_mask[idx] = True

for idx in test_ids:
    test_mask[idx] = True

In [34]:
data["disease"]["train_mask"] = train_mask
data["disease"]["val_mask"] = val_mask
data["disease"]["test_mask"] = test_mask

In [35]:
#Hetero Data
print(data)

HeteroData(
  disease={
    x=[20163, 200],
    y=[20163],
    train_mask=[20163],
    val_mask=[20163],
    test_mask=[20163],
  },
  gene={ x=[13561, 200] },
  chemical={ x=[26522, 200] },
  species={ x=[2863, 200] },
  (gene, to, gene)={ edge_index=[2, 43] },
  (gene, to, disease)={ edge_index=[2, 54] },
  (disease, to, disease)={ edge_index=[2, 68] },
  (chemical, to, gene)={ edge_index=[2, 59] },
  (chemical, to, disease)={ edge_index=[2, 97] },
  (chemical, to, chemical)={ edge_index=[2, 122] },
  (chemical, to, species)={ edge_index=[2, 13] },
  (species, to, gene)={ edge_index=[2, 5] },
  (species, to, disease)={ edge_index=[2, 11] },
  (species, to, species)={ edge_index=[2, 1] }
)


In [36]:
# This code works in torch-geometric==2.6.0
homo_data = data.to_homogeneous(add_edge_type=False)

In [37]:
homo_data

Data(edge_index=[2, 473], x=[63109, 200], y=[63109], train_mask=[63109], val_mask=[63109], test_mask=[63109], node_type=[63109])

In [38]:
homo_data.x = torch.rand(homo_data.x.shape[0], 8)
homo_data

Data(edge_index=[2, 473], x=[63109, 8], y=[63109], train_mask=[63109], val_mask=[63109], test_mask=[63109], node_type=[63109])

In [39]:
homo_data.x[0]

tensor([0.3508, 0.8184, 0.1939, 0.4723, 0.4583, 0.7327, 0.0383, 0.6440])

In [40]:
torch.save(homo_data, '../../data/pubmed/processed/data.pt')

In [41]:
torch.unique(homo_data.y)

tensor([-1,  0,  1,  2,  3,  4,  5,  6,  7])

In [42]:
homo_data.has_isolated_nodes()

True

In [43]:
homo_data.has_self_loops()

True